# gold_dec_fec

Camada **Gold** de continuidade do fornecimento: DEC e FEC por alimentador e mês, para a tela de Operação.

**O que são DEC e FEC.** São os dois indicadores que a ANEEL usa para medir a qualidade do fornecimento de energia:
- **DEC** é quantas horas, em média, cada consumidor ficou sem energia no período. DEC 2,5 no mês significa que cada UC do alimentador ficou, em média, 2h30 sem energia.
- **FEC** é quantas vezes, em média, cada consumidor ficou sem energia. FEC 0,5 no mês significa, em média, uma interrupção a cada duas UCs por mês.

O DEC mede quanto tempo a energia faltou, e o FEC mede quantas vezes. Para um alimentador num mês, DEC = soma das horas de interrupção ÷ UCs do alimentador, e FEC = número de interrupções ÷ UCs do alimentador. Como cada linha da Silver já é uma UC interrompida, basta somar as durações e contar as linhas.

**O que este notebook faz.** Resume as interrupções da Silver numa tabela com uma linha por alimentador e mês. A tela de Operação e a IA leem essa tabela pronta para mostrar os piores alimentadores, a evolução no tempo e a comparação entre municípios.

- **Entrada:** `gold.dim_mes`, `gold.dim_alimentador`, `gold.fato_alimentador_mes` e `gold.dim_uc` (`gold_base`) e `silver.tb_ocorrencias_rede` (`silver_rede_atendimento`).
- **Saída:** `gold.agg_dec_fec_mensal`.
- **Alimenta:** `GET /operacao` (série histórica, ranking de alimentadores e ranking regional) e o agente de IA.
- Segue o documento *Decisões da Engenharia de Dados: camada Gold* (v2, seções 4 e 5.1). Idempotente: grava em overwrite e pode ser reexecutado.

| Tabela | Grão (1 linha por) | Conteúdo |
|---|---|---|
| `agg_dec_fec_mensal` | alimentador × mês (122 × 48) | município, período (P1/P2), UCs atendidas no mês, interrupções, UCs interrompidas, horas de interrupção (CHI), DEC e FEC com e sem dia crítico e contagens das flags da Silver |

**Regras**
- **Fórmula (PRODIST Módulo 8):** DEC = Σ t(i) ÷ Cc e FEC = Σ Ca(i) ÷ Cc. Cada linha da Silver é uma UC interrompida, então Ca(i) = 1 e t(i) = `duracao_efetiva_min` em horas. Cc = `ucs_atendidas` do alimentador **no mês** (`fato_alimentador_mes`: UCs ligadas até o fim do mês).
- **O que conta como interrupção:** duração de 3 minutos ou mais (interrupção longa do PRODIST). Ficam fora, mas continuam na Silver: OSCILACAO (`flag_sem_interrupcao`; variação de tensão, a energia não falta) e as ocorrências abertas antes da ligação da UC (`flag_antes_ligacao`; a UC ainda não existia).
- **Sobreposição:** o DEC usa a `duracao_efetiva_min`, que desconta o trecho que a mesma UC já estava sem energia. O FEC continua contando cada interrupção.
- **Dia crítico:** `dec` e `fec` incluem tudo. `dec_sem_dia_critico` e `fec_sem_dia_critico` tiram os dias com `flag_dia_critico` (dia com interrupções acima da média + 3 desvios), como o expurgo do PRODIST. Na base, são os 31 dias de julho de 2025.
- **Mês pela data de abertura:** a interrupção que vira o mês fica inteira no mês em que abriu (regra combinada).
- **`horas_interrupcao`** é o CHI da ANEEL (consumidor-hora interrompido): ajuda a ver onde mais gente sofreu, já que o DEC favorece alimentador pequeno. **`ucs_interrompidas`** conta as UCs distintas interrompidas no mês.
- **Mês sem interrupção** tem 0 interrupções e DEC/FEC 0. Não é dado faltando.
- **Mesma definição no `gold_kpis`:** o DEC/FEC global do `gold_kpis` precisa usar estas mesmas regras para continuar batendo com a soma dos alimentadores.
- Antes de gravar: unicidade do grão, chaves sem nulos, grade completa, flags que deixariam o DEC errado zeradas e reconciliação com a Silver. Se algo falhar, nada é gravado.

## Funções comuns
**O que fazemos:** carregamos as funções compartilhadas pelo time.

In [0]:
%run ../bronze-to-silver/funcoes_comuns

## Configuração
**O que fazemos:** lemos o catálogo do widget.

In [0]:
dbutils.widgets.text("catalog", "workspace")
CATALOGO = dbutils.widgets.get("catalog")

## Parâmetros
**O que fazemos:** reunimos os parâmetros do cálculo, cada um com o que representa e o seu valor no contexto real. Mudar uma regra é mudar só esta célula.

In [0]:
# Duração mínima para a interrupção contar no DEC/FEC.
# Real: o PRODIST chama de interrupção longa a que dura 3 minutos ou mais.
# Nos dados, a menor interrupção tem 5 minutos, então nenhuma sai, mas a regra fica no código.
DURACAO_MINIMA_DEC_MIN = 3

# Minutos por hora: o DEC é medido em horas por UC, e a Silver guarda a duração em minutos.
MINUTOS_POR_HORA = 60

# Grão da tabela: 1 linha por alimentador e mês.
CHAVE = ["alimentador", "mes_referencia"]

# Flags da Silver que tiram a ocorrência do DEC/FEC (a linha continua na Silver).
# Real: OSCILACAO é variação de tensão, a energia não falta (decisão do time, 23/10);
# antes da ligação, a UC ainda não existia (erro de dado).
FLAGS_EXCLUI = ["flag_sem_interrupcao", "flag_antes_ligacao"]

# Flags que precisam estar zeradas: se aparecerem, o DEC ficaria errado sem aviso, então o notebook para.
FLAGS_ZERADAS = ["flag_alimentador_inconsistente", "flag_duracao_inconsistente", "flag_encerramento_antes_abertura"]

# Contagens das flags da Silver expostas por alimentador e mês (entram no cálculo, ficam visíveis à parte).
COLUNAS_FLAGS = ["ocorrencias_sobrepostas", "ocorrencias_teto_48h", "ocorrencias_acima_24h", "ocorrencias_dia_critico"]

# Ordem final das colunas (nomes iguais aos do contrato da API).
COLUNAS = [*CHAVE, "municipio", "periodo", "ucs_atendidas", "interrupcoes", "ucs_interrompidas",
           "horas_interrupcao", "dec", "fec", "dec_sem_dia_critico", "fec_sem_dia_critico", *COLUNAS_FLAGS]

## Leitura
**O que fazemos:** lemos as dimensões e o Cc por mês da `gold_base` e as ocorrências da Silver.

In [0]:
dim_mes = spark.table(f"{CATALOGO}.gold.dim_mes").select("mes_referencia", "periodo")
dim_alim = spark.table(f"{CATALOGO}.gold.dim_alimentador").select("alimentador", "municipio", "ucs_atendidas")
cc_mes = spark.table(f"{CATALOGO}.gold.fato_alimentador_mes").select(*CHAVE, "ucs_atendidas")   # Cc de cada mês
oc = spark.table(f"{CATALOGO}.silver.tb_ocorrencias_rede")

N_MESES, N_ALIM, N_OC = dim_mes.count(), dim_alim.count(), oc.count()
N_UCS = spark.table(f"{CATALOGO}.gold.dim_uc").select("id_uc").count()
print(f"meses: {N_MESES} | alimentadores: {N_ALIM} | UCs: {N_UCS:,} | ocorrências na Silver: {N_OC:,}")

**O que descobrimos:**
- As entradas têm o tamanho esperado: 48 meses, 122 alimentadores, 45.000 UCs e 180.000 ocorrências na Silver.

## Pré-condições das entradas
**O que fazemos:** conferimos que as dimensões e o Cc por mês estão completos antes de calcular. O Cc de cada alimentador em cada mês não pode ser zero ou nulo, e a soma dos Cc de um mês não pode passar do total de UCs do cadastro. Se algo falhar, o problema está na `gold_base`, e não aqui.

In [0]:
checar_unicidade(dim_alim, ["alimentador"], "dim_alimentador")
checar_unicidade(dim_mes, ["mes_referencia"], "dim_mes")
checar_unicidade(cc_mes, CHAVE, "fato_alimentador_mes")
checar_sem_nulos(dim_alim, ["alimentador", "municipio", "ucs_atendidas"], "dim_alimentador")
checar_sem_nulos(cc_mes, [*CHAVE, "ucs_atendidas"], "fato_alimentador_mes")

soma_cc = dim_alim.agg(F.sum("ucs_atendidas")).first()[0]
assert soma_cc == N_UCS, f"Σ ucs_atendidas = {soma_cc:,}, mas a dim_uc tem {N_UCS:,}"

n_cc = cc_mes.count()
assert n_cc == N_ALIM * N_MESES, f"fato_alimentador_mes com {n_cc:,} linhas, esperado {N_ALIM * N_MESES:,}"
sem_ucs = cc_mes.filter(F.col("ucs_atendidas") <= 0).count()
assert sem_ucs == 0, f"{sem_ucs} pares alimentador × mês com Cc zero: o DEC deles seria uma divisão por zero"
cc_por_mes = cc_mes.groupBy("mes_referencia").agg(F.sum("ucs_atendidas").alias("cc"))
cc_min, cc_max = cc_por_mes.agg(F.min("cc"), F.max("cc")).first()
assert cc_max <= N_UCS, f"Σ Cc de um mês ({cc_max:,}) passa das {N_UCS:,} UCs da dim_uc"

print(f"ok: {N_ALIM} alimentadores, Cc de {cc_mes.agg(F.min('ucs_atendidas')).first()[0]} "
      f"a {cc_mes.agg(F.max('ucs_atendidas')).first()[0]} UCs | Cc total do mês: de {cc_min:,} a {cc_max:,}")

**O que descobrimos:**
- As dimensões estão completas: alimentadores e meses únicos, sem nulos.
- O Cc por alimentador varia de 20 a 2.285 UCs. Na distribuidora, o Cc do mês sobe de 44.644 UCs (jul/2022) para 45.000 (a partir de mar/2026), conforme as UCs são ligadas. Nenhum alimentador tem Cc zero em nenhum mês.

## Ocorrências válidas
**O que fazemos:** conferimos que as flags que deixariam o DEC errado estão zeradas. Depois ficamos só com as interrupções de 3 minutos ou mais, sem OSCILACAO e sem as anteriores à ligação da UC, e definimos o mês pela data de abertura, como no `gold_kpis`.

In [0]:
# Proteção: estas flags são 0 hoje; se aparecerem, o DEC ficaria errado sem aviso
zeradas = oc.agg(*[F.sum(F.col(f).cast("long")).alias(f) for f in FLAGS_ZERADAS]).first().asDict()
assert not any(zeradas.values()), f"flags que deixariam o DEC errado: {zeradas}"

curta = F.col("duracao_min") < DURACAO_MINIMA_DEC_MIN
exclui = F.array_contains(F.array(*FLAGS_EXCLUI), True)
oc_validas = (oc.filter(~curta & ~exclui)
                .withColumn("mes_referencia", F.date_format("data_abertura", "yyyy-MM")))
N_VALIDAS = oc_validas.count()

fora = oc.agg(F.sum(curta.cast("long")).alias(f"abaixo de {DURACAO_MINIMA_DEC_MIN} min"),
              *[F.sum(F.col(f).cast("long")).alias(f) for f in FLAGS_EXCLUI]).first().asDict()
print(f"ocorrências válidas: {N_VALIDAS:,} de {N_OC:,} | fora do DEC/FEC: {N_OC - N_VALIDAS:,}")
print(f"motivos (uma ocorrência pode ter mais de um): {fora}")

**O que descobrimos:**
- Das 180.000 ocorrências da Silver, 164.892 entram no DEC/FEC. Saem 14.562 oscilações e 595 ocorrências anteriores à ligação da UC (49 têm os dois motivos). Nenhuma sai pela duração: a menor tem 5 minutos.
- As flags que deixariam o DEC errado (alimentador diferente do cadastro, duração inconsistente e encerramento antes da abertura) estão zeradas.

## Integridade com as dimensões
**O que fazemos:** conferimos que todo alimentador e todo mês das ocorrências existem nas dimensões. Se algum faltar, aquelas interrupções sumiriam do DEC sem aviso.

In [0]:
alim_fora = oc_validas.join(dim_alim, "alimentador", "left_anti")
mes_fora = oc_validas.join(dim_mes, "mes_referencia", "left_anti")
n_alim_fora, n_mes_fora = alim_fora.count(), mes_fora.count()

if n_alim_fora:
    alim_fora.groupBy("alimentador").count().show(truncate=False)
if n_mes_fora:
    mes_fora.groupBy("mes_referencia").count().show(truncate=False)
assert n_alim_fora == 0, f"{n_alim_fora:,} ocorrências com alimentador fora da dim_alimentador"
assert n_mes_fora == 0, f"{n_mes_fora:,} ocorrências com mês fora da dim_mes"
print("ok: todo alimentador e todo mês das ocorrências existem nas dimensões")

**O que descobrimos:**
- Todo alimentador e todo mês das ocorrências existem nas dimensões. Nenhuma interrupção fica de fora do cálculo.

## Agregação por alimentador e mês
**O que fazemos:** somamos, para cada alimentador e mês, o número de interrupções e os minutos efetivos de interrupção, com e sem os dias críticos, e contamos as UCs interrompidas. Também contamos as interrupções marcadas pelas flags da Silver. As métricas ficam num dicionário só, usado na agregação e na reconciliação.

In [0]:
conta = lambda flag: F.sum(F.col(flag).cast("long"))
fora_dia_critico = ~F.col("flag_dia_critico")

# Interrupção que fica inteira dentro de outra da mesma UC (duração efetiva 0) é o mesmo apagão
# registrado duas vezes: não conta no DEC (já não somava minutos) nem no FEC
conta_no_fec = F.col("duracao_efetiva_min") > 0

# Métricas somáveis: as mesmas expressões agregam por alimentador e mês e reconciliam com a Silver
METRICAS = {
    "interrupcoes": F.sum(conta_no_fec.cast("long")),
    "__minutos": F.sum("duracao_efetiva_min"),
    "__minutos_sem_dc": F.sum(F.when(fora_dia_critico, F.col("duracao_efetiva_min")).otherwise(0)),
    "__interrupcoes_sem_dc": F.sum((conta_no_fec & fora_dia_critico).cast("long")),
    "ocorrencias_sobrepostas": conta("flag_sobreposicao"),
    "ocorrencias_teto_48h": conta("flag_duracao_teto"),
    "ocorrencias_acima_24h": conta("flag_longa_duracao"),
    "ocorrencias_dia_critico": conta("flag_dia_critico"),
}

agregado = (oc_validas.groupBy(*CHAVE)
    .agg(*[expr.alias(nome) for nome, expr in METRICAS.items()],
         F.countDistinct("id_uc").alias("ucs_interrompidas")))
print(f"pares alimentador × mês com interrupção: {agregado.count():,} de {N_ALIM * N_MESES:,} possíveis")
print(f"interrupções inteiras dentro de outra (fora do FEC): {oc_validas.filter(~conta_no_fec).count():,}")

**O que descobrimos:**
- Todos os 5.856 pares de alimentador e mês têm pelo menos uma interrupção. A grade completa não precisa preencher nenhum mês com zero, mas a regra fica no código. As 147 interrupções que ficam inteiras dentro de outra da mesma UC não contam no FEC.

## Grade completa e cálculo do DEC e FEC
**O que fazemos:** cruzamos todos os alimentadores com todos os meses, para que mês sem interrupção apareça com 0, e trazemos o Cc de cada mês. Depois calculamos as horas, o DEC e o FEC, com e sem dia crítico, sempre a partir dos minutos (sem arredondar antes) e com `try_divide`.

In [0]:
SOMAS = [*METRICAS, "ucs_interrompidas"]
por_cc = lambda c, fator=1: F.try_divide(F.col(c), F.col("ucs_atendidas") * fator)

dec_fec = (dim_alim.drop("ucs_atendidas").crossJoin(dim_mes)
    .join(cc_mes, CHAVE)
    .join(agregado, CHAVE, "left")
    .fillna(0, subset=SOMAS)
    .withColumns({
        "horas_interrupcao": F.try_divide(F.col("__minutos"), F.lit(MINUTOS_POR_HORA)).cast("decimal(14,2)"),
        "dec": por_cc("__minutos", MINUTOS_POR_HORA).cast("decimal(14,4)"),
        "fec": por_cc("interrupcoes").cast("decimal(14,4)"),
        "dec_sem_dia_critico": por_cc("__minutos_sem_dc", MINUTOS_POR_HORA).cast("decimal(14,4)"),
        "fec_sem_dia_critico": por_cc("__interrupcoes_sem_dc").cast("decimal(14,4)"),
    })
    .select(*COLUNAS, "__minutos", "__minutos_sem_dc", "__interrupcoes_sem_dc"))

## Checagens e reconciliação
**O que fazemos:** conferimos o grão, os nulos e a grade completa. A reconciliação garante que as interrupções, os minutos (com e sem dia crítico) e as flags da Gold são iguais aos das ocorrências válidas da Silver. Se algo falhar, nada é gravado.

In [0]:
checar_unicidade(dec_fec, CHAVE, "agg_dec_fec_mensal")
checar_sem_nulos(dec_fec, [*CHAVE, "municipio", "ucs_atendidas", "interrupcoes", "ucs_interrompidas",
                           "horas_interrupcao", "dec", "fec", "dec_sem_dia_critico", "fec_sem_dia_critico"],
                 "agg_dec_fec_mensal")

n_linhas = dec_fec.count()
assert n_linhas == N_ALIM * N_MESES, f"esperado {N_ALIM * N_MESES:,} linhas, veio {n_linhas:,}"

g = dec_fec.agg(*[F.sum(c).alias(c) for c in METRICAS]).first().asDict()
s = oc_validas.agg(*[expr.alias(nome) for nome, expr in METRICAS.items()]).first().asDict()
assert g == s, f"reconciliação não fecha: gold={g} silver={s}"

print(f"ok: {n_linhas:,} linhas | interrupções {g['interrupcoes']:,} | horas {g['__minutos'] / MINUTOS_POR_HORA:,.2f} "
      f"(sem dia crítico {g['__minutos_sem_dc'] / MINUTOS_POR_HORA:,.2f}) | sobrepostas {g['ocorrencias_sobrepostas']} | "
      f"teto 48h {g['ocorrencias_teto_48h']} | acima 24h {g['ocorrencias_acima_24h']} | "
      f"dia crítico {g['ocorrencias_dia_critico']:,}: igual à Silver")

**O que descobrimos:**
- O grão é único, as chaves não têm nulos e a tabela tem as 5.856 linhas esperadas.
- A reconciliação fecha com a Silver: 164.745 interrupções (as 147 inteiras dentro de outra ficam fora do FEC), 330.918,07 horas efetivas (244.309,90 sem os dias críticos), 322 sobrepostas, 29 no teto de 48h, 265 acima de 24h e 17.825 em dia crítico.

## Gravação
**O que fazemos:** gravamos a tabela em overwrite, com a coluna de rastreio `_atualizado_em`.

In [0]:
(dec_fec.drop("__minutos", "__minutos_sem_dc", "__interrupcoes_sem_dc")
        .withColumn("_atualizado_em", F.current_timestamp())
        .write.format("delta").mode("overwrite").option("overwriteSchema", "true")
        .saveAsTable(f"{CATALOGO}.gold.agg_dec_fec_mensal"))

## Conferência da tabela gravada
**O que fazemos:** lemos a tabela de volta e conferimos o número de linhas, os tipos e uma amostra.

In [0]:
gold = spark.table(f"{CATALOGO}.gold.agg_dec_fec_mensal")
print(f"gold.agg_dec_fec_mensal: {gold.count():,} linhas")
gold.printSchema()
gold.orderBy("alimentador", "mes_referencia").show(5, truncate=False)

**O que descobrimos:**
- A tabela foi gravada com 5.856 linhas e os tipos do padrão da Gold: DEC e FEC em decimal(14,4), horas em decimal(14,2) e contagens inteiras.
- Além das colunas de antes, ela traz `ucs_interrompidas`, `dec_sem_dia_critico`, `fec_sem_dia_critico` e `ocorrencias_dia_critico`, e o `ucs_atendidas` agora é o Cc do mês.

## DEC e FEC global por período
**O que fazemos:** somamos os alimentadores para obter o DEC e o FEC da distribuidora em cada mês (Σ horas ÷ Σ Cc do mês) e, no período, somamos os meses, como no contrato da API. Mostramos também o resultado sem os dias críticos. Usamos os minutos ainda sem arredondar.

In [0]:
global_mes = (dec_fec.groupBy("mes_referencia", "periodo")
    .agg(F.sum("__minutos").alias("minutos"), F.sum("__minutos_sem_dc").alias("minutos_sem_dc"),
         F.sum("interrupcoes").alias("interrupcoes"), F.sum("__interrupcoes_sem_dc").alias("interrupcoes_sem_dc"),
         F.sum("ucs_atendidas").alias("cc"))
    .withColumns({"dec_global": F.col("minutos") / (F.col("cc") * MINUTOS_POR_HORA),
                  "fec_global": F.col("interrupcoes") / F.col("cc"),
                  "dec_sem_dia_critico": F.col("minutos_sem_dc") / (F.col("cc") * MINUTOS_POR_HORA),
                  "fec_sem_dia_critico": F.col("interrupcoes_sem_dc") / F.col("cc")}))
INDICES = ["dec_global", "fec_global", "dec_sem_dia_critico", "fec_sem_dia_critico"]

print("== DEC e FEC global por período (soma dos meses)")
(global_mes.filter(F.col("periodo").isNotNull()).groupBy("periodo")
     .agg(F.sum("interrupcoes").alias("interrupcoes"), F.round(F.sum("minutos") / MINUTOS_POR_HORA).alias("horas"),
          *[F.round(F.sum(c), 4).alias(c) for c in INDICES])
     .orderBy("periodo").show(truncate=False))

print("== DEC e FEC global por mês")
(global_mes.select("mes_referencia", "cc", "interrupcoes", *[F.round(c, 4).alias(c) for c in INDICES])
     .orderBy("mes_referencia").show(48, truncate=False))

**O que descobrimos:**
- Um mês normal tem DEC de cerca de 0,11 h/UC e FEC de cerca de 0,068. Cada UC fica, em média, uns 7 minutos sem energia por mês.
- Do P1 para o P2, o DEC global sobe de 1,34 para 3,30 h/UC (+146%) e o FEC de 0,81 para 1,23 (+51%).
- Quase toda a piora é julho de 2025, o mês dos 31 dias críticos, com DEC de 1,93 h/UC (17 vezes um mês normal). Sem os dias críticos, o P2 tem DEC de 1,37 (+2,2% sobre o P1) e FEC de 0,83 (+2,1%). Ou seja, fora o evento extremo, a qualidade do fornecimento ficou estável.
- Novembro de 2025 e fevereiro de 2026 também aparecem acima do normal (DEC de cerca de 0,18), mas nenhum dia deles passa do limite de dia crítico.
- Em relação à versão anterior (todas as ocorrências, Cc fixo e duração original), o DEC cai cerca de 9% no P1 (1,47 → 1,34) e 4% no P2 (3,45 → 3,30), quase tudo pela saída das oscilações.

## Piores e melhores alimentadores
**O que fazemos:** montamos o ranking dos últimos 12 meses (P2), que é o que a tela de Operação mostra por padrão. Listamos os 10 piores e os 5 melhores em DEC, os 10 piores em FEC e os 10 piores em horas de interrupção (CHI), que mostra onde mais consumidores sofreram, sem favorecer alimentador pequeno.

In [0]:
ranking_p2 = (gold.filter(F.col("periodo") == "P2")
    .groupBy("alimentador", "municipio")
    .agg(F.max("ucs_atendidas").alias("ucs_atendidas"),
         F.sum("interrupcoes").alias("interrupcoes"),
         F.sum("horas_interrupcao").alias("horas_interrupcao"),
         F.sum("dec").alias("dec_p2"), F.sum("dec_sem_dia_critico").alias("dec_p2_sem_dia_critico"),
         F.sum("fec").alias("fec_p2")))

print("== 10 piores em DEC (P2)")
ranking_p2.orderBy(F.desc("dec_p2")).show(10, truncate=False)
print("== 5 melhores em DEC (P2)")
ranking_p2.orderBy("dec_p2").show(5, truncate=False)
print("== 10 piores em FEC (P2)")
ranking_p2.orderBy(F.desc("fec_p2")).show(10, truncate=False)
print("== 10 piores em horas de interrupção, o CHI (P2)")
ranking_p2.orderBy(F.desc("horas_interrupcao")).show(10, truncate=False)

**O que descobrimos:**
- Os 7 piores em DEC continuam sendo alimentadores muito pequenos, de 20 a 47 UCs, com 335 a 365 interrupções no ano. O pior, ALM-JAB-016, tem 20 UCs e DEC de 40,4 h/UC.
- Os três alimentadores do evento de julho (ALM-SAO-004, ALM-JAB-004 e ALM-REC-008) têm DEC de cerca de 15 h/UC. Sem os dias críticos, caem para 0,2 a 0,6 h/UC: o DEC deles é quase todo julho.
- No ranking por horas de interrupção (CHI), o quadro muda. No topo ficam os três alimentadores do evento (34 mil, 17 mil e 13 mil horas) e, depois deles, alimentadores médios, de 96 a 578 UCs, com cerca de 820 a 890 horas cada. Os pequenos somam poucas horas.
- Os melhores em DEC são alimentadores grandes, com 979 a 1.338 UCs. O ranking de FEC repete os pequenos.

## Concentração das interrupções
**O que fazemos:** verificamos se as interrupções acompanham o tamanho do alimentador. Se não acompanharem, alimentadores pequenos ficam com DEC alto só por terem poucas UCs no denominador. Também medimos o peso de julho de 2025 no DEC de cada período.

In [0]:
total_4_anos = (gold.groupBy("alimentador")
                    .agg(F.max("ucs_atendidas").alias("ucs_atendidas"),
                         F.sum("interrupcoes").alias("interrupcoes"), F.sum("dec").alias("dec_total")))

print("correlação entre UCs do alimentador e interrupções (4 anos):",
      round(total_4_anos.stat.corr("ucs_atendidas", "interrupcoes"), 3))
print("correlação entre UCs do alimentador e DEC (4 anos):",
      round(total_4_anos.stat.corr("ucs_atendidas", "dec_total"), 3))

print("\n== interrupções por faixa de tamanho do alimentador (4 anos)")
(total_4_anos
    .withColumn("faixa_ucs", F.when(F.col("ucs_atendidas") < 100, "1. menos de 100")
                              .when(F.col("ucs_atendidas") < 300, "2. 100 a 299")
                              .when(F.col("ucs_atendidas") < 600, "3. 300 a 599")
                              .otherwise("4. 600 ou mais"))
    .groupBy("faixa_ucs")
    .agg(F.count("*").alias("alimentadores"),
         F.round(F.avg("ucs_atendidas")).alias("ucs_media"),
         F.round(F.avg("interrupcoes")).alias("interrupcoes_media"),
         F.round(F.avg("dec_total"), 2).alias("dec_medio_4_anos"))
    .orderBy("faixa_ucs").show(truncate=False))

print("== peso de julho de 2025 no DEC global de P2")
p2 = global_mes.filter(F.col("periodo") == "P2")
dec_p2 = p2.agg(F.sum("dec_global")).first()[0]
dec_jul = p2.filter(F.col("mes_referencia") == "2025-07").first()["dec_global"]
dec_p2_sem_dc = p2.agg(F.sum("dec_sem_dia_critico")).first()[0]
print(f"DEC P2: {dec_p2:.4f} h/UC | só julho de 2025: {dec_jul:.4f} ({dec_jul / dec_p2:.0%} do período) | "
      f"P2 sem dia crítico: {dec_p2_sem_dc:.4f}")

**O que descobrimos:**
- Fora os três alimentadores do evento de julho, o número de interrupções quase não depende do tamanho do alimentador: cerca de 1.250 em 4 anos, tanto nos de 60 UCs quanto nos de 440. A correlação positiva (0,62) vem desses três alimentadores grandes.
- Como o DEC divide pelas UCs, os alimentadores pequenos ficam com DEC muito mais alto: 47,8 h/UC em 4 anos nos menores, contra cerca de 5,4 nos maiores (correlação de −0,41).
- Julho de 2025 sozinho responde por 58% do DEC global do P2. Sem os dias críticos, o DEC do P2 é 1,37 h/UC.
- Para a tela de Operação, vale mostrar o DEC junto com as horas de interrupção (CHI) e com o DEC sem dia crítico: o DEC sozinho destaca alimentadores pequenos e o evento de julho.

## Resumo
**O que entregamos:** a tabela `gold.agg_dec_fec_mensal`, com DEC e FEC de cada um dos 122 alimentadores em cada um dos 48 meses (5.856 linhas), com e sem os dias críticos. Ela alimenta a tela de Operação e o agente de IA.

**Como garantimos que está certa:**
- As entradas estão completas e o Cc de cada alimentador e mês vem da `fato_alimentador_mes`, sem zeros.
- Entram 164.892 das 180.000 ocorrências: saem as oscilações e as anteriores à ligação da UC, que continuam na Silver com flag. A reconciliação fecha com a Silver em interrupções, horas (com e sem dia crítico) e flags.
- O DEC usa a duração efetiva: o tempo em que a UC já estava sem energia não conta duas vezes.
- O grão é único, sem nulos, e o cálculo segue o PRODIST. O `gold_kpis` precisa das mesmas regras para continuar batendo.

**O que os números mostram:**
- Um mês normal tem DEC de cerca de 0,11 h/UC. Do P1 para o P2, o DEC global sobe 146%, mas 58% do DEC do P2 vem de julho de 2025. Sem os dias críticos, a qualidade fica estável (+2,2%).
- As interrupções se distribuem quase igualmente entre os alimentadores, sem acompanhar o tamanho deles. Por isso, os piores em DEC são alimentadores muito pequenos (20 a 47 UCs); pelo CHI, os piores são os três do evento de julho e alimentadores médios.

**Decisões registradas (checkup de 23/10):** OSCILACAO fora do DEC/FEC; ocorrências anteriores à ligação fora; duração efetiva (322 sobrepostas); Cc por mês; colunas sem dia crítico; interrupção de 3 minutos ou mais; a interrupção que vira o mês fica inteira no mês de abertura (cerca de 460 casos, um quarto deles em julho de 2025).